# Dialog Open Delay — Investigation Notebook

**Problem**: Three dialog types open with an erratic 0–12 second delay.

1. Dialogs/Finished/Preview dialog (`ShowPreviewDialog` in `dialogs.cpp`)
2. Entry screen → Open Project dialog (`OpenProjectDlgProc` in `main.cpp`)
3. License dialog from About page (`ShowLicenseDialog` in `about.cpp`)

The delay is **erratic** (not constant), which points to disk-cache-state effects or
first-time COM/DLL initialisation rather than deterministic CPU work.

This notebook documents the investigation step by step and will be updated
after each code change.  Conclusions will be copied into `dialogs_INTERNALS.txt`.

## 1 — Affected code paths (where is time spent?)

### 1a. About / License dialogs: `GdiplusStartup` per call

```cpp
// about.cpp — ShowAboutDialog (called every time the About dialog opens)
GdiplusStartupInput gdiplusStartupInput;   // SuppressBackgroundThread = FALSE (default)
ULONG_PTR gdiplusToken;
GdiplusStartup(&gdiplusToken, &gdiplusStartupInput, NULL);  // ← creates a COM background thread
g_logoImage = Image::FromFile(logoPath);                    // ← reads SetupCraft.png from disk
// … dialog runs …
GdiplusShutdown(gdiplusToken);                              // ← tears down the thread
```

**Why this is slow (and erratic)**

| Call | Cost (cold) | Cost (warm) |
|---|---|---|
| `GdiplusStartup` (SuppressBackgroundThread=FALSE) | 200 ms – several seconds<br>(creates COM thread, loads gdiplus.dll) | ~0 ms (already in proc) |
| `Image::FromFile` (SetupCraft.png) | 50–500 ms (disk I/O) | ~1 ms (OS file cache) |
| `GdiplusShutdown` | tears down thread → next open re-pays full init cost | — |

GDI+ is shut down on every close and re-initialised on every open.  
On a cold disk or after memory pressure, each open can cost 2–5+ seconds.

**Fix**: Initialise GDI+ **once** in `wWinMain` with `SuppressBackgroundThread = TRUE`.
Remove `GdiplusStartup` / `GdiplusShutdown` from `ShowAboutDialog`.
Rely on process termination to clean up GDI+ (acceptable for an exe — no
DLL-unload constraints).

**Status**: ✅ Fixed in commit after this notebook was created.

### 1b. Preview dialog: `StreamRtfIn` in `WM_CREATE` (before window is shown)

```cpp
// dialogs.cpp — PreviewWndProc WM_CREATE
const std::wstring& rtf = s_dialogs[(int)pd->type].content_rtf;
if (!rtf.empty()) {
    StreamRtfIn(pd->hContent, rtf);   // ← EM_STREAMIN synchronous in WM_CREATE
}
```

`WM_CREATE` runs *inside* `CreateWindowExW`, so the window is not yet visible
while `StreamRtfIn` runs.  If the RTF is large (e.g. the License page stores the
full GPL v2 as RTF), EM_STREAMIN must parse and format the entire document before
`CreateWindowExW` returns and the window can be shown.

**Observed worst case**: 0–2 s on a warm machine for typical installer dialog RTF.

**After-window delay**: The first `WM_PAINT` of the visible RichEdit (fired in the
message loop) renders already-formatted RTF text to screen.  For plain/simple RTF
this is fast.  For RTF with many embedded images or OLE objects it can be 9+ s
(see inline comment in `ShowPreviewDialog`).
→ Currently `RDW_UPDATENOW` is NOT used, so this paint is fully asynchronous and
is only slow when message-loop `WM_PAINT` runs.

**Status**: Under investigation.

### 1c. Preview dialog: `UpdateWindow(hPreview)` before `AutoFitPreview`

```cpp
// dialogs.cpp — ShowPreviewDialog (and NavigateTo)
ShowWindow(hPreview, SW_SHOW);
UpdateWindow(hPreview);       // ← forces sync WM_PAINT of hPreview
AutoFitPreview(hPreview, &pd);
```

The old intent was to give `MeasureRichEditLogHeight` a "genuine paint pass" so
that `GetScrollInfo` would return a non-zero scroll range.  However, `AutoFitPreview`
now always creates a **hidden** (no `WS_VISIBLE`) 1-px measurement RichEdit, whose
`GetScrollInfo` always returns 0 → falls through to `EM_FORMATRANGE` (measure-only).
The measurement is therefore independent of any paint pass on the visible RichEdit.

`UpdateWindow(hPreview)` sends `WM_PAINT` to the **parent** window only, not to its
child RichEdit.  So even if the RichEdit's first paint were slow, `UpdateWindow`
on the parent would not trigger it.

**Conclusion**: `UpdateWindow(hPreview)` is a no-op for the current measurement
path and does not cause or fix the delay.  It is a harmless remnant.

**Status**: No change needed; note added to the comment in `AutoFitPreview`.

### 1d. Open Project dialog: `SpawnSelf` is the perceived delay source

```cpp
// main.cpp — OpenProjectDlgProc WM_COMMAND IDOK
SpawnSelf(arg);          // CreateProcessW — loads SetupCraft.exe + all MinGW DLLs
g_projectOpened = true;
DestroyWindow(hDlg);
```

`SpawnSelf` launches a fresh copy of `SetupCraft.exe` with `CreateProcessW`.  On a
**cold disk** (after system restart or if the working set was evicted):

| File | HDD cold load | SSD cold load |
|---|---|---|
| SetupCraft.exe | 0.5–3 s | <100 ms |
| mingw_dlls/*.dll (libgcc, libstdc++, etc.) | 1–5 s each | <200 ms each |

Total worst-case on a spinning HDD: 5–12 seconds.  On an SSD it is usually < 0.5 s.

The Open Project **dialog itself** opens fast (< 50 ms — WM_CREATE only creates
a ListView and two custom buttons; icon extraction happens lazily at first paint
via `WM_DRAWITEM`).

**Fix options**:
- Not easily fixable at the SpawnSelf level without major architecture change.
- Consider pre-reading the exe/dlls from a background thread during the Open
  Project dialog display time (readahead), so the OS file cache is warm by
  the time the user clicks OK.
- Alternatively, accept the delay on cold HDD as a hardware limitation and
  document it.

**Status**: Under investigation / documented.

### 1e. License dialog: `LoadLibraryW(Riched20.dll)` and file I/O

```cpp
// about.cpp — ShowLicenseDialog
LoadLibraryW(L"Riched20.dll");       // loads RichEdit DLL if not already loaded
// ...
HBITMAP hLogoBitmap = (HBITMAP)LoadImageW(NULL, logoPath.c_str(), IMAGE_BITMAP,
    0, 0, LR_LOADFROMFILE);          // reads GnuLogo.bmp from disk
```

`Riched20.dll` is almost always already loaded by the time the License dialog opens
(the Preview dialog loads Msftedit.dll / Riched20.dll in its WM_CREATE), so the
`LoadLibraryW` call is a no-op (returns cached handle).

`GnuLogo.bmp` is a small bitmap; on any modern disk it loads in < 10 ms.

The `AppendRichText` calls inside `ShowLicenseDialog` run synchronously after
`CreateWindowExW` (with `WS_VISIBLE`), so the window frame appears immediately but
the RichEdit content streams in before the message loop starts.  For the GPL v2
text this takes 100–300 ms.

The perceived License dialog delay is almost entirely the **About dialog** delay
(GdiplusStartup) since the user must open About first, then click License.
Once the About fix (#1a) is in place, the License dialog should feel instant.

**Status**: Fixed indirectly by fix #1a.

## 2 — Root cause summary

| Dialog | Root cause | Severity | Fixed? |
|---|---|---|---|
| About / License | `GdiplusStartup` called on every open with background thread | **High** | ✅ |
| Open Project (perceived) | `SpawnSelf` → `CreateProcessW` cold DLL load | Medium | Documented |
| Preview (initial show) | `StreamRtfIn` in `WM_CREATE` before window visible | Low–Medium | Under investigation |
| Preview (first paint) | RichEdit `WM_PAINT` with large/complex RTF | Medium | Mitigated by no `RDW_UPDATENOW` |

The erratic 0–12 s range is explained by OS disk-cache state:
- **Warm** (files in page cache): < 100 ms for most operations → 0 s perceived
- **Cold** (after restart or memory pressure on HDD): full disk load → up to 12 s

SSD systems typically stay under 1 s even cold.

## 3 — Fix log

### Fix A: Move `GdiplusStartup` to app startup

**Files changed**: `main.cpp`, `about.cpp`

**`main.cpp`** — added at the top of `wWinMain` (after `CoInitializeEx`):
```cpp
Gdiplus::GdiplusStartupInput gsi = {};
gsi.SuppressBackgroundThread = TRUE;
ULONG_PTR gdiplusToken = 0;
Gdiplus::GdiplusStartup(&gdiplusToken, &gsi, NULL);
```

**`about.cpp`** — removed from `ShowAboutDialog`:
```cpp
// REMOVED:
GdiplusStartupInput gdiplusStartupInput;
ULONG_PTR gdiplusToken;
GdiplusStartup(&gdiplusToken, &gdiplusStartupInput, NULL);
// ...and at close...
GdiplusShutdown(gdiplusToken);
```

**Why `SuppressBackgroundThread = TRUE`**:
SetupCraft does not use GDI+ notification callbacks.  The background thread is
only needed for applications that use `Image::GetThumbnailImage` with callbacks
or for notification of metafile changes.  Suppressing it avoids thread creation
overhead.

**Why no `GdiplusShutdown`**:
GDI+ cleans up on process exit.  Calling `GdiplusShutdown` is only required
when unloading GDI+ from a DLL while the host process continues.  For an exe
that exits completely, relying on process termination is safe and avoids
complicating the multi-return-path structure of `wWinMain`.

**Commit**: TBD

## 4 — Text for `dialogs_INTERNALS.txt`

The following block is ready to paste as a new section at the end of `dialogs_INTERNALS.txt`:

```
════════════════════════════════════════════════════════════════════════════════
Dialog open delay — root causes and mitigations
════════════════════════════════════════════════════════════════════════════════

Symptom: Erratic 0–12 s delay before dialogs appear or respond.
The variance is explained by OS disk-cache state (warm = 0 s, cold HDD = 12 s).

── About / License dialog: GdiplusStartup per call ─────────────────────────
CAUSE:
  ShowAboutDialog called GdiplusStartup(SuppressBackgroundThread=FALSE) on every
  open and GdiplusShutdown on every close.  GdiplusStartup with the default
  startup input spawns a COM background thread.  On a cold system or after
  memory pressure this thread creation + gdiplus.dll load takes 2–5+ seconds.
  Repeated open/close cycles re-paid this cost each time.

FIX (applied):
  GdiplusStartup is called ONCE in wWinMain with SuppressBackgroundThread=TRUE.
  No GdiplusShutdown is called (process exit provides cleanup — safe for exe).
  ShowAboutDialog no longer calls GdiplusStartup or GdiplusShutdown.
  Image::FromFile(SetupCraft.png) still runs each open (reloads the PNG);
  on warm cache this is ~1 ms and is acceptable.

── Preview dialog: StreamRtfIn in WM_CREATE ────────────────────────────────
CAUSE:
  PreviewWndProc WM_CREATE calls StreamRtfIn(pd->hContent, rtf) synchronously.
  WM_CREATE runs inside CreateWindowExW, so the window is not yet visible.
  For large RTF (e.g. License page with full GPL v2 text) EM_STREAMIN must
  parse and reflow the entire document before the window appears — 0.5–2 s.

STATUS: Under investigation.  Possible fix: defer StreamRtfIn via PostMessage
  from WM_CREATE so window appears immediately, then content loads asynchronously.
  Requires matching deferral of AutoFitPreview.

── Preview dialog: first WM_PAINT with complex RTF ─────────────────────────
CAUSE:
  The RichEdit first-paint renders the fully-formatted document to the screen DC.
  For RTF with many embedded images or OLE objects this can take 9+ seconds.
  IMPORTANT: RDW_UPDATENOW must NOT be used in ShowPreviewDialog (it would force
  a synchronous paint immediately from the same thread).  Use RDW_INVALIDATE only
  so that WM_PAINT is queued and the message loop processes it on its next pass.

NOTE on UpdateWindow(hPreview) in ShowPreviewDialog / NavigateTo:
  This call sends WM_PAINT to the parent window only, not to child controls.
  AutoFitPreview uses a hidden (no WS_VISIBLE) 1-px measurement RichEdit and
  always falls through to EM_FORMATRANGE (GetScrollInfo returns 0 on hidden
  windows).  The UpdateWindow call is therefore harmless but unnecessary.

── Open Project dialog: perceived delay from SpawnSelf ─────────────────────
CAUSE:
  The dialog itself opens fast (WM_CREATE only creates ListView + custom buttons;
  icon extraction is lazy at first WM_DRAWITEM).  The perceived delay is from
  SpawnSelf("--open=<id>") in the IDOK handler: CreateProcessW loads
  SetupCraft.exe + all MinGW DLLs from disk.  On cold HDD: 5–12 s.

STATUS: Architectural limitation.  No fix applied.  Acceptable on SSD.
  Future option: readahead the exe/dlls in a background thread while the
  dialog is displayed so the OS page cache is warm when the user clicks OK.
```